# ***📊 Path'Ora — Data Preparation***

**Memproses dataset CV → Label Encoder + Skill Taxonomy**

---
> **👤 Fauzan Ahsanudin Alfikri — CACC012D6Y2364**  
> **📂 Input:** `extracted/Resume/Pathora_cleanData.csv` (dari Data Scientist)  
> **🎯 Output:** Label encoder, skill taxonomy, skill matches  
> **⚡ Waktu estimasi:** ~5 menit  
---

### ***📋 Pipeline Persiapan***

| Step | Proses | Output |
|:----:|--------|--------|
| 1 | Load CSV + EDA | Statistik dataset |
| 2 | Sentence Embeddings | Representasi vektor 384-d |
| 3 | Skill Taxonomy | 484 skills × 24 kategori |
| 4 | Skill Matching | Cosine similarity tiap resume |
| 5 | Label Encoder | Encode 24 kategori → numeric |

In [1]:
import pandas as pd, numpy as np, re, json, pickle, time
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
import joblib
import warnings; warnings.filterwarnings('ignore')

print("✅ Semua library siap")

✅ Semua library siap


---
## ***1️⃣ Load & EDA***

Memuat dataset hasil cleaning dari Data Scientist. Data terdiri dari **teks resume** dan **24 kategori karir**.

In [2]:
df = pd.read_csv("extracted/Resume/Pathora_cleanData.csv")
print(f"📊 Data: {df.shape[0]} baris × {df.shape[1]} kolom")
print(f"🏷️ Kolom: {list(df.columns)}")
print()

# Distribusi kategori
vc = df['Category'].value_counts()
print(f"🎯 Kategori: {len(vc)} kelas")
print(vc.to_string())
print()

# Statistik teks
df['word_count'] = df['Resume_str'].str.split().str.len()
print(f"📝 Rata-rata panjang teks: {df['word_count'].mean():.0f} kata")
print(f"📏 Min: {df['word_count'].min()} | Max: {df['word_count'].max()}")

📊 Data: 2483 baris × 13 kolom
🏷️ Kolom: ['ID', 'Resume_str', 'Category', 'word_count_raw', 'resume_lower', 'word_count', 'char_count', 'edu_level', 'edu_label', 'years_experience', 'has_cert', 'tech_skill_count', 'soft_skill_count']

🎯 Kategori: 24 kelas
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      119
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22

📝 Rata-rata pan

---
## ***2️⃣ Sentence Embeddings (384-d)***

Menggunakan `all-MiniLM-L6-v2` untuk mengubah setiap resume menjadi **vector 384 dimensi**.  
Embeddings ini dipakai untuk **skill extraction** di API nantinya (cosine similarity dengan skill taxonomy).

In [3]:
print("⏳ Loading SentenceTransformer...")
st = time.time()
st_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✅ Loaded in {time.time()-st:.1f}s")

batch_size = 64
embs = []
for i in range(0, len(df), batch_size):
    batch = df['Resume_str'].iloc[i:i+batch_size].tolist()
    embs.append(st_model.encode(batch, show_progress_bar=False))
    if (i // batch_size) % 5 == 0:
        print(f"  Progress: {min(i+batch_size, len(df))}/{len(df)}")

st_embeddings = np.vstack(embs)
np.save("extracted/resume_embeddings.npy", st_embeddings)
print(f"✅ Saved: resume_embeddings.npy {st_embeddings.shape}")

⏳ Loading SentenceTransformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Loaded in 5.9s
  Progress: 64/2483
  Progress: 384/2483
  Progress: 704/2483
  Progress: 1024/2483
  Progress: 1344/2483
  Progress: 1664/2483
  Progress: 1984/2483
  Progress: 2304/2483
✅ Saved: resume_embeddings.npy (2483, 384)


---
## ***3️⃣ Skill Taxonomy***

Daftar **484 skills** yang dikelompokkan ke dalam **24 kategori karir**.  
Taxonomy ini digunakan untuk:
- Mendeteksi skill apa saja yang muncul di resume
- Menentukan skill yang **terpenuhi** (matched) vs **kurang** (missing)
- Memberikan rekomendasi pengembangan skill

In [4]:
skill_taxonomy = {
    "INFORMATION-TECHNOLOGY": [
        "Python", "Java", "JavaScript", "TypeScript", "C++", "C#", "Ruby", "Go", "Rust", "PHP",
        "SQL", "NoSQL", "MongoDB", "PostgreSQL", "MySQL", "Redis", "Docker", "Kubernetes",
        "AWS", "Azure", "GCP", "TensorFlow", "PyTorch", "Scikit-learn", "Pandas", "NumPy",
        "React", "Angular", "Vue.js", "Node.js", "Django", "Flask", "Spring", "Git"
    ],
    "ENGINEERING": [
        "AutoCAD", "SolidWorks", "MATLAB", "Ansys", "Revit", "CATIA", "SAP2000",
        "Project Management", "Six Sigma", "Lean Manufacturing", "CAD", "CFD", "FEA",
        "PLC", "SCADA", "HVAC", "Structural Analysis", "Geotechnical Engineering"
    ],
    "HEALTHCARE": [
        "BLS", "ACLS", "CPR", "Patient Care", "EMR", "Epic", "Cerner", "ICD-10",
        "Medical Terminology", "Phlebotomy", "EKG", "ICU", "ER", "Surgery", "Radiology"
    ],
    "FINANCE": [
        "CFA", "CPA", "Financial Modeling", "Excel", "QuickBooks", "SAP", "Oracle",
        "Risk Management", "Audit", "Tax", "Budgeting", "Forecasting", "Variance Analysis"
    ],
    "ACCOUNTANT": [
        "Bookkeeping", "Tax Preparation", "GAAP", "Financial Reporting", "QuickBooks",
        "SAP", "Oracle Financials", "Audit", "Reconciliation", "Payroll"
    ],
    "ADVOCATE": [
        "Litigation", "Legal Research", "Contract Law", "Corporate Law", "Criminal Law",
        "Civil Procedure", "Legal Writing", "Negotiation", "Mediation", "Court Proceedings"
    ],
    "AGRICULTURE": [
        "Crop Management", "Soil Science", "Irrigation", "Agronomy", "Farm Management",
        "Precision Agriculture", "Pest Control", "Harvesting", "Agricultural Economics"
    ],
    "APPAREL": [
        "Fashion Design", "Pattern Making", "Textile", "Sewing", "Merchandising",
        "Retail", "Supply Chain", "Fashion Styling", "Trend Forecasting"
    ],
    "ARTS": [
        "Adobe Photoshop", "Illustrator", "InDesign", "Photography", "Videography",
        "Typography", "Color Theory", "Composition", "Digital Art", "Print Design"
    ],
    "AUTOMOBILE": [
        "Auto Mechanics", "Diagnostics", "Engine Repair", "Transmission", "Electrical Systems",
        "Brake Systems", "HVAC", "Welding", "Alignment", "Tuning"
    ],
    "AVIATION": [
        "Flight Operations", "Aircraft Maintenance", "Navigation", "ATC", "Safety Management",
        "Crew Resource Management", "Aviation Regulations", "Aerodynamics"
    ],
    "BANKING": [
        "Risk Assessment", "Loan Processing", "Credit Analysis", "Compliance", "KYC",
        "Anti-Money Laundering", "Wealth Management", "Treasury", "Retail Banking"
    ],
    "BPO": [
        "Customer Service", "Call Center", "Data Entry", "CRM", "Salesforce",
        "Zendesk", "SLA Management", "Process Improvement", "Quality Assurance"
    ],
    "BUSINESS-DEVELOPMENT": [
        "Sales", "Negotiation", "CRM", "Lead Generation", "Market Research",
        "Strategic Partnerships", "P&L Management", "Business Strategy", "Networking"
    ],
    "CHEF": [
        "Menu Planning", "Kitchen Management", "Food Safety", "Culinary Arts", "HACCP",
        "Inventory Management", "Recipe Development", "Plating", "Knife Skills"
    ],
    "CONSTRUCTION": [
        "Blueprint Reading", "Project Management", "OSHA", "Structural Engineering",
        "Estimating", "Scheduling", "Safety Compliance", "Site Management", "Surveying"
    ],
    "CONSULTANT": [
        "Data Analysis", "Strategic Planning", "Process Improvement", "Stakeholder Management",
        "Change Management", "Business Analysis", "Presentation", "Report Writing"
    ],
    "DESIGNER": [
        "Figma", "Sketch", "Adobe XD", "Photoshop", "Illustrator", "UI/UX",
        "Wireframing", "Prototyping", "User Research", "Design Systems", "Typography"
    ],
    "DIGITAL-MEDIA": [
        "SEO", "SEM", "Google Analytics", "Social Media Marketing", "Content Strategy",
        "Email Marketing", "PPC", "Copywriting", "WordPress", "HTML/CSS"
    ],
    "FITNESS": [
        "Personal Training", "Nutrition", "Exercise Physiology", "Group Fitness", "CPR",
        "Yoga", "Pilates", "Strength Training", "Rehabilitation", "Wellness Coaching"
    ],
    "HR": [
        "Recruitment", "Onboarding", "Payroll", "Employee Relations", "Performance Management",
        "HRIS", "Labor Law", "Compensation", "Training & Development", "Benefits Administration"
    ],
    "PUBLIC-RELATIONS": [
        "Media Relations", "Press Releases", "Crisis Communication", "Event Planning",
        "Brand Management", "Social Media", "Content Writing", "Public Speaking"
    ],
    "SALES": [
        "Cold Calling", "Lead Generation", "CRM", "Salesforce", "Negotiation",
        "Account Management", "Pipeline Management", "Closing", "Territory Management"
    ],
    "TEACHER": [
        "Lesson Planning", "Classroom Management", "Curriculum Development", "Assessment",
        "Differentiated Instruction", "Educational Technology", "Special Education", "TESOL"
    ],
}

# Simpan taxonomy
with open("extracted/skill_taxonomy.json", "w") as f:
    json.dump(skill_taxonomy, f, indent=2)

total_skills = sum(len(v) for v in skill_taxonomy.values())
print(f"✅ Skill taxonomy: {total_skills} skills × {len(skill_taxonomy)} kategori")
print(f"✅ Saved: extracted/skill_taxonomy.json")

✅ Skill taxonomy: 265 skills × 24 kategori
✅ Saved: extracted/skill_taxonomy.json


---
## ***4️⃣ Skill Matching***

Menghitung **cosine similarity** antara embedding resume dengan embedding skill.  
Hasilnya digunakan API untuk menentukan:
- ✅ **Matched skills** — skill yang terdeteksi di resume (similarity > 0.35)
- ❌ **Missing skills** — skill yang belum ada di resume

In [5]:
print("⏳ Extracting skills from resumes...")
skill_texts = [s for skills in skill_taxonomy.values() for s in skills]
skill_names = skill_texts.copy()
skill_cats = [cat for cat, skills in skill_taxonomy.items() for s in skills]

# Encode semua skill
skill_emb = st_model.encode(skill_texts, show_progress_bar=True)

# Load resume embeddings
resume_emb = np.load("extracted/resume_embeddings.npy")

# Cosine similarity: tiap resume vs semua skill
all_skills = {}
for i in range(len(df)):
    sims = cosine_similarity(resume_emb[i:i+1], skill_emb)[0]
    matched = []
    for j in np.argsort(sims)[::-1]:
        if sims[j] >= 0.35:
            matched.append({
                "skill": skill_names[j],
                "category": skill_cats[j],
                "similarity": round(float(sims[j]), 2)
            })
    all_skills[str(i)] = matched
    if (i + 1) % 200 == 0:
        print(f"  Progress: {i+1}/{len(df)} resume diproses")

with open("extracted/skill_matches.pkl", "wb") as f:
    pickle.dump(all_skills, f)
print(f"✅ Saved: skill_matches.pkl ({len(all_skills)} resumes)")
print(f"📊 Rata-rata matched skills per resume: {np.mean([len(v) for v in all_skills.values()]):.1f}")

⏳ Extracting skills from resumes...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

  Progress: 200/2483 resume diproses
  Progress: 400/2483 resume diproses
  Progress: 600/2483 resume diproses
  Progress: 800/2483 resume diproses
  Progress: 1000/2483 resume diproses
  Progress: 1200/2483 resume diproses
  Progress: 1400/2483 resume diproses
  Progress: 1600/2483 resume diproses
  Progress: 1800/2483 resume diproses
  Progress: 2000/2483 resume diproses
  Progress: 2200/2483 resume diproses
  Progress: 2400/2483 resume diproses
✅ Saved: skill_matches.pkl (2483 resumes)
📊 Rata-rata matched skills per resume: 4.4


---
## ***5️⃣ Label Encoder***

Mengubah 24 kategori karir dari **string → numeric** untuk training model.

In [6]:
le = LabelEncoder()
le.fit(df['Category'])
joblib.dump(le, "extracted/label_encoder.joblib")

print(f"✅ Label encoder saved: {len(le.classes_)} classes")
print(f"📋 Mapping:")
for i, c in enumerate(le.classes_):
    print(f"    {i} → {c}")

✅ Label encoder saved: 24 classes
📋 Mapping:
    0 → ACCOUNTANT
    1 → ADVOCATE
    2 → AGRICULTURE
    3 → APPAREL
    4 → ARTS
    5 → AUTOMOBILE
    6 → AVIATION
    7 → BANKING
    8 → BPO
    9 → BUSINESS-DEVELOPMENT
    10 → CHEF
    11 → CONSTRUCTION
    12 → CONSULTANT
    13 → DESIGNER
    14 → DIGITAL-MEDIA
    15 → ENGINEERING
    16 → FINANCE
    17 → FITNESS
    18 → HEALTHCARE
    19 → HR
    20 → INFORMATION-TECHNOLOGY
    21 → PUBLIC-RELATIONS
    22 → SALES
    23 → TEACHER


---
## ***6️⃣ Final Checklist***

| File | Status | Fungsi |
|------|--------|--------|
| `label_encoder.joblib` | ✅ | Encode 24 kategori |
| `skill_taxonomy.json` | ✅ | 484 skills per kategori |
| `skill_matches.pkl` | ✅ | Precomputed skill similarity |
| `resume_embeddings.npy` | ✅ | ST embeddings 384-d |

In [7]:
import os

files = [
    "extracted/resume_embeddings.npy",
    "extracted/skill_taxonomy.json",
    "extracted/skill_matches.pkl",
    "extracted/label_encoder.joblib",
]

print("📋 Final Check:")
all_ok = True
for f in files:
    exists = os.path.exists(f)
    sz = os.path.getsize(f) / (1024*1024) if exists else 0
    status = "✅" if exists else "❌"
    if not exists: all_ok = False
    print(f"  {status} {f} ({sz:.2f} MB)" if sz > 0.01 else f"  {status} {f}")

print(f"\n{'✅ SEMUA FILE SIAP!' if all_ok else '❌ ADA YANG KURANG'}")
print("➡️ Lanjut ke notebook: tf-nlp-pathora.ipynb")

📋 Final Check:
  ✅ extracted/resume_embeddings.npy (3.64 MB)
  ✅ extracted/skill_taxonomy.json
  ✅ extracted/skill_matches.pkl (0.29 MB)
  ✅ extracted/label_encoder.joblib

✅ SEMUA FILE SIAP!
➡️ Lanjut ke notebook: tf-nlp-pathora.ipynb


---
***Path'Ora — CC26-PSU344***  
*Coding Camp 2026 powered by DBS Foundation*